In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
import os
# Define the local folder filename
csv_filename = 'home_depot_sales_data.csv'
# Read the CSV file from the specified path
full_path = os.path.join(r'C:\Users\tobia\OneDrive\Desktop\School\advanced_business_analytics\data', csv_filename)
df = pd.read_csv(full_path)
df.head()


,month,sku_name,sku_num,region,sales_units
0,2024-04-01,CPVC-GLD-0.5-PIPE [2FT],206041,PACIFIC_NORTHWEST,2.418678
1,2025-09-01,CPVC-GLD-0.5-PIPE [2FT],206041,PACIFIC_NORTHWEST,12.006578
2,2025-02-01,CPVC-GLD-0.5-PIPE [2FT],206041,PACIFIC_NORTHWEST,4.480735
3,2025-04-01,CPVC-GLD-0.5-PIPE [2FT],206041,PACIFIC_NORTHWEST,7.926598
4,2024-12-01,CPVC-GLD-0.5-PIPE [2FT],206041,PACIFIC_NORTHWEST,4.778459


In [ ]:
target_col = 'sales_units'

# version 1
df_model = df.copy()
df_model["month"] = pd.to_datetime(df_model["month"])
df_model = df_model.sort_values("month").reset_index(drop=True)

prophet_df = df_model[["month", target_col]].rename(columns={"month": "ds", target_col: "y"})

# Train/test split (70/30)
split_idx = int(len(prophet_df) * 0.7)
train = prophet_df.iloc[:split_idx].copy()
test = prophet_df.iloc[split_idx:].copy()

# Fit Prophet model
model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    seasonality_mode="additive"
)
model.fit(train)

# Predict on test set
test_forecast = model.predict(test[["ds"]])
test[f"predicted_{target_col}"] = test_forecast["yhat"].values

# Metrics
rmse = np.sqrt(mean_squared_error(test["y"], test[f"predicted_{target_col}"]))
r2 = r2_score(test["y"], test[f"predicted_{target_col}"])

print("Test RMSE:", rmse)
print("Test R^2:", r2)

# Refit on full dataset for 24-month forecast
final_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    seasonality_mode="additive"
)
final_model.fit(prophet_df)

future = final_model.make_future_dataframe(periods=24, freq="MS")
forecast = final_model.predict(future)

future_forecast = forecast[forecast["ds"] > prophet_df["ds"].max()][["ds", "yhat"]].copy()
future_forecast = future_forecast.rename(
    columns={"ds": "month", "yhat": f"predicted_{target_col}"}
)
future_forecast["sku_name"] = 'CPVC-GLD-0.5-PIPE [2FT]'
future_forecast["sales_unit"] = "sales units"

print(future_forecast[["month", "sku_name", "sales_unit", f"predicted_{target_col}"]])

# Plot
plt.figure(figsize=(12, 6))
plt.plot(prophet_df["ds"], prophet_df["y"], label="Actual")
plt.plot(test["ds"], test[f"predicted_{target_col}"], label="Test Prediction")
plt.plot(
    future_forecast["month"],
    future_forecast[f"predicted_{target_col}"],
    label="24-Month Forecast"
)
plt.xlabel("Month")
plt.ylabel(target_col)
plt.title(f"Prophet Forecast with Seasonality - {'CPVC-GLD-0.5-PIPE [2FT]'} {target_col}")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()